### QICK Standard ZCU111 Firmware + FFT/Spectrum Analysis

This firmware is a standard ZCU111 firmware (with less SGv6 though 
to make room for the BREAD Small logic) + BREAD Small chain. 
The BREAD small chain is a reduced FFT version 
(16k in place of 32k points) of the BREAD Real project.

BREAD Small part: this part has one input ADC and one output DAC. 

ADC is connected to Tile 224 Channel 0. DAC is connected to Tile 229 Channel 2.

There's an option to connect ADC_224_0 to a Readout path to capture 
data with the tProcessor.

See examples to extract data from the different points.

<img src="images/qick_diagrams-QICK_TPROCV2_111_FFT_SPECTRUM.drawio.png" alt="Placeholder" width="1200" height="1000">


## Aims
* Enumerate notebooks and tests to verify the ZCU111 board and the Spectrum firmware works as expected.
* Give ZCU111 users a set of tests to report any issues to the QICK team.
* Show most of the available data resources in the signal path and ways to analyze them.

----

## 1. ADC raw data acquire

In this section we analize data captured by the ADC and stored in the **BUFFER_ADC** buffer.

<img src="images/buffer_adc.png" alt="Placeholder" width="200" height="150">

Please run the included notebook to verify your raw ADC data. 

[ADC raw data analysis](./01_ADC_raw_data.ipynb)

----

## 2. PFB data analysis

In this section we analize data captured and stored in the **BUFFER_PFB** buffer, with the **DDS_CIC** block bypassed.

<img src="images/buffer_pfb.png" alt="Placeholder" width="200" height="150">

Please run the included Polyphase Filter Bank (PFB) notebook to verify/analize your data after crossing the PFB.  

[PFB data analysis](./02_PFB_data.ipynb)

----

## 3. DDS + CIC data analysis

In this section we analize data captured by the ADC and stored in the **BUFFER_ADC** buffer with the **DDS_CIC** block enabled.

<img src="images/buffer_pfb.png" alt="Placeholder" width="200" height="150">

Please run the included DDS + CIC notebook to verify/analyze your data after crossing the DDS+CIC IP block.

[DDS + CIC data analysis](./03_DDSCIC_data.ipynb)

----

## 4. WXFFT_64k data analysis

In this section we analize data captured by the ADC and stored in the **ACCUMULATOR_1** buffer with the **DDS_CIC** block enabled.

<img src="images/acc_1.png" alt="Placeholder" width="200" height="150">

Please run the included DDS + CIC notebook to verify/analyze your data after crossing the DDS+CIC IP block.

[WXFFT_64k data analysis](./04_WXFFT_data.ipynb)

----

## Summary

The above notebooks and tests were selected to help users
verify the correctness of acquired data in each signal stage after entering into the processing path of the Spectrum firmware.

For questions related to these notebooks or other functionality
with QICK or this board - please visit the QICK github page at
[openquantumhardware.com](https://github.com/openquantumhardware). 

----

## Last revised

* 10 Mar 2026 - Initial Revision

---

----

----

----

# Appendix: MSB Alignment in 16-bit Registers

## Important Notice

Since the 12-bit data in the RFSoC ADC is **MSB-aligned in a 16-bit register**, the data is essentially shifted left by 4 bits ($16 - 12 = 4$). This fundamental characteristic affects all calculations throughout the signal processing chain and must be understood to correctly interpret measurement results.

---

## 1. The Mapping Physics

In the FPGA registers, the 12-bit signed ADC value ($D_{12}$) relates to the 16-bit value ($D_{16}$) as follows:

$$D_{16} = D_{12} \times 2^4 = D_{12} \times 16$$

**Example:**
* Native 12-bit ADC code: `1000` (decimal)
* Stored in 16-bit FPGA register: `10000000000000` (16 bits total, 12 bits shifted left, 4 zero bits on right)
* Decimal value: $1000 \times 16 = 16,000$

---

## 2. Impact on the System Parameters

Because of this alignment, the values entering the FFT are 16 times larger than the "native" 12-bit ADC counts. We must update our reference values accordingly:

### Full-Scale (FS) Reference

Instead of the "native" 12-bit full scale of $2^{11} = 2,048$, the FPGA logic sees:

$$FS_{FPGA} = 2^{11} \times 2^4 = 2^{15} = 32,768$$

This $2^{15}$ reference is used for **all dBFS calculations** throughout the system.

### FFT Processing Gain

Since the FFT (Xilinx/AMD IP) has **no internal scaling**, the output magnitude grows by a factor of $N$ (the FFT size). For $N = 65,536$:

* **Voltage gain**: $N = 65,536$
* **Power gain**: $N^2$ (when converting amplitude to power)

### Power Calculation and dB Offset

Since power is proportional to the square of the amplitude, the 4-bit shift ($2^4$) results in a **power scaling of $(2^4)^2 = 2^8 = 256$**, which equals:

$$20 \log_{10}(16) = 24 \text{ dB}$$

This **+24 dB offset** appears in all power calculations downstream relative to a standard 12-bit alignment.

---

## 3. dBFS Definition in This System

With MSB alignment, **0 dBFS corresponds to a full-scale complex sinusoid at the ADC input**:

$$\text{0 dBFS} \leftrightarrow \text{amplitude} = 32,768 \text{ (in 16-bit FPGA register units)}$$

Common reference levels:
* **0 dBFS** → amplitude = 32,768
* **−6.02 dBFS** → amplitude = 16,384 (half-scale)
* **−20 dBFS** → amplitude ≈ 3,276

This $FS = 32,768$ reference is maintained consistently throughout all downstream DSP blocks (PFB, DDS+CIC, FFT).

---

## 4. Implications for All Downstream Calculations

Every section of the analysis chain must account for this alignment:

| Stage | Effect |
|-------|--------|
| **ADC Reading** | 12-bit data becomes 16-bit after left-shift by 4 |
| **Voltage Conversion** | $V = \text{ADC\_code} \times V_{LSB}$ uses the 16-bit value |
| **RMS Calculation** | RMS is computed from the 16-bit scaled values |
| **FFT Processing** | FFT operates on 16-bit data, magnitude output includes $N$ gain |
| **dBFS Measurement** | All dBFS values referenced to FS = 32,768 |
| **Calibration Constants** | K_ADC, C_FFT constants measured with this 32,768 reference |

---

## 5. Summary Table

| Parameter | Value | Formula |
|-----------|-------|---------|
| ADC Resolution | 12 bits | Native |
| FPGA Register Width | 16 bits | MSB-aligned |
| Left-shift Factor | 4 bits | 16 − 12 |
| Effective Full-Scale | 32,768 | $2^{15}$ |
| dB Offset vs. native 12-bit | +24 dB | $20 \log_{10}(16)$ |

---